# AlphaGo (Silver et al., 2016) — Implementation / 구현

**Paper**: Silver, D., Huang, A., Maddison, C. J., et al. (2016). *Mastering the game of Go with deep neural networks and tree search.* **Nature, 529**(7587), 484–489. DOI: [10.1038/nature16961](https://doi.org/10.1038/nature16961)

## Scope / 범위

**English**: We cannot reproduce AlphaGo itself — it required 30M KGS positions, 40 search threads, and 176 GPUs. Instead we build a **pedagogical miniature** that demonstrates every algorithmic idea in the paper on **Tic-Tac-Toe** (a 3×3 perfect-information zero-sum game). All four ingredients are present:

1. **Pure MCTS with UCT** — the classical baseline.
2. **PUCT with a policy prior** — the AlphaGo-style selection rule.
3. **Value network + mixed leaf evaluation** — $V(s_L) = (1-\lambda) v_\theta + \lambda z_L$.
4. **Policy-gradient self-play (REINFORCE)** — stage-2 of the training pipeline.

We then reproduce the paper's headline ablation — the **$\lambda$ mixing sweep from Fig. 4b** — in miniature, showing that the mixed evaluation dominates either pure evaluator.

**한국어**: AlphaGo 자체는 3천만 KGS 국면과 176 GPU가 필요해 그대로 재현할 수 없다. 대신 **Tic-Tac-Toe** (3×3 완전정보 제로섬 게임)에서 논문의 네 가지 알고리즘 아이디어를 모두 구현한 **교육용 축소판** 을 만든다.

1. **순수 MCTS (UCT)** — 고전적 baseline
2. **PUCT + 정책 prior** — AlphaGo 방식의 선택 규칙
3. **가치망 + 혼합 리프 평가** — $V(s_L) = (1-\lambda) v_\theta + \lambda z_L$
4. **Policy gradient self-play (REINFORCE)** — 학습 파이프라인의 2단계

마지막으로 논문의 핵심 ablation — **Fig 4b의 $\lambda$ 혼합 스윕** — 을 축소판으로 재현한다.

In [ ]:
import math
import random
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Part 1: Tic-Tac-Toe Environment / Tic-Tac-Toe 환경

**English**: A minimal zero-sum perfect-information environment. Board is a length-9 array with $+1$ (X), $-1$ (O), $0$ (empty). Current player is a separate flag. Terminal states return $\pm 1$ for win/loss, $0$ for draw. This plays the role of the 19×19 Go board in our miniature.

**한국어**: 최소한의 제로섬 완전정보 환경. 보드는 길이 9 배열로 $+1$(X), $-1$(O), $0$(빈칸). 현재 플레이어는 별도 플래그. 종단 상태는 승 $+1$ / 패 $-1$ / 무승부 $0$. 축소판에서 19×19 바둑판의 역할을 한다.

In [ ]:
WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]

@dataclass(frozen=True)
class State:
    """Immutable Tic-Tac-Toe state.

    Attributes:
        board: tuple of 9 ints in {-1, 0, +1}.
        player: +1 (X) or -1 (O), whose turn it is.
    """
    board: tuple
    player: int

    @staticmethod
    def initial() -> "State":
        return State(board=tuple([0] * 9), player=+1)

    def legal_moves(self) -> list:
        return [i for i, c in enumerate(self.board) if c == 0]

    def apply(self, move: int) -> "State":
        new = list(self.board)
        new[move] = self.player
        return State(board=tuple(new), player=-self.player)

    def winner(self) -> Optional[int]:
        """Return +1 / -1 if a player has three in a row, 0 if draw, None if ongoing."""
        for a, b, c in WIN_LINES:
            s = self.board[a] + self.board[b] + self.board[c]
            if s == 3:
                return +1
            if s == -3:
                return -1
        if 0 not in self.board:
            return 0
        return None

    def is_terminal(self) -> bool:
        return self.winner() is not None

    def to_tensor(self) -> torch.Tensor:
        """Return a 3-channel 3x3 tensor: own stones, opponent stones, turn plane."""
        arr = np.array(self.board, dtype=np.float32).reshape(3, 3)
        own = (arr == self.player).astype(np.float32)
        opp = (arr == -self.player).astype(np.float32)
        turn = np.full_like(own, 1.0 if self.player == +1 else 0.0)
        return torch.from_numpy(np.stack([own, opp, turn]))  # (3, 3, 3)

    def render(self) -> str:
        sym = {+1: 'X', -1: 'O', 0: '.'}
        rows = [' '.join(sym[c] for c in self.board[i:i+3]) for i in (0, 3, 6)]
        return '\n'.join(rows)

s0 = State.initial()
print('Initial:')
print(s0.render())
print('Legal moves:', s0.legal_moves())
s1 = s0.apply(4).apply(0).apply(2)
print('\nAfter X-centre, O-corner, X-corner:')
print(s1.render())
print('Tensor shape:', s0.to_tensor().shape)

## Part 2: Pure MCTS with UCT / 순수 MCTS (UCT)

**English**: Classical MCTS (Kocsis & Szepesvári 2006). At each internal node we select the child that maximises

$$ \arg\max_a\ Q(s,a) + c\sqrt{\tfrac{\ln N(s)}{N(s,a)}} $$

with $c=\sqrt{2}$. Rollouts use uniform random play. This is the baseline — no neural networks.

**한국어**: 고전 MCTS. 각 내부 노드에서 위 식을 최대화하는 자식을 선택. $c=\sqrt{2}$. 롤아웃은 균등 랜덤 플레이. Baseline — 신경망 없음.

In [ ]:
@dataclass
class Node:
    """A node in the MCTS tree.

    Statistics are stored per-edge on the parent side, encoded as dicts keyed by action.
    """
    state: State
    parent: Optional["Node"] = None
    move_from_parent: Optional[int] = None
    children: dict = field(default_factory=dict)
    N: int = 0
    W: float = 0.0
    prior: dict = field(default_factory=dict)  # action -> prior probability

    @property
    def Q(self) -> float:
        return self.W / self.N if self.N > 0 else 0.0

    def is_expanded(self) -> bool:
        return len(self.children) > 0


def uct_select(node: Node, c: float = math.sqrt(2)) -> Node:
    """Select a child via the UCT formula."""
    logN = math.log(node.N + 1)
    best, best_val = None, -math.inf
    for child in node.children.values():
        if child.N == 0:
            val = math.inf  # force at least one visit
        else:
            val = -child.Q + c * math.sqrt(logN / child.N)  # negate: opponent perspective
        if val > best_val:
            best, best_val = child, val
    return best


def expand(node: Node):
    """Expand all children with a uniform prior."""
    moves = node.state.legal_moves()
    uniform = 1.0 / len(moves) if moves else 0.0
    for m in moves:
        child_state = node.state.apply(m)
        node.children[m] = Node(state=child_state, parent=node, move_from_parent=m)
        node.prior[m] = uniform


def random_rollout(state: State) -> int:
    """Play uniformly random moves until terminal. Return outcome from +1's perspective."""
    s = state
    while not s.is_terminal():
        s = s.apply(random.choice(s.legal_moves()))
    return s.winner()


def backup(node: Node, value: float):
    """Propagate the value up the tree, flipping sign each ply."""
    cur = node
    v = value
    while cur is not None:
        cur.N += 1
        cur.W += v
        v = -v
        cur = cur.parent


def mcts_uct(root_state: State, n_simulations: int = 500) -> int:
    """Run UCT MCTS and return the most-visited root action."""
    root = Node(state=root_state)
    expand(root)
    for _ in range(n_simulations):
        node = root
        # 1) Selection
        while node.is_expanded() and not node.state.is_terminal():
            node = uct_select(node)
        # 2) Expansion
        if not node.state.is_terminal():
            expand(node)
        # 3) Simulation
        outcome_from_x = random_rollout(node.state) if not node.state.is_terminal() else node.state.winner()
        # Convert outcome to perspective of the node's player-to-move
        value = outcome_from_x * node.state.player
        # 4) Backup
        backup(node, value)
    # Choose most-visited child
    return max(root.children.items(), key=lambda kv: kv[1].N)[0]


# Quick sanity check: MCTS as X on an empty board should choose centre or a corner
move = mcts_uct(State.initial(), n_simulations=2000)
print(f'UCT-MCTS opening move (2000 sims): cell {move}  ', {4: 'CENTRE', 0: 'CORNER', 2: 'CORNER', 6: 'CORNER', 8: 'CORNER'}.get(move, 'EDGE'))

## Part 3: Policy & Value Networks / 정책망과 가치망

**English**: Small CNNs mirroring AlphaGo's shapes but on a 3×3 grid with 3 input channels. The policy head outputs a distribution over 9 cells (softmax-masked by legal moves); the value head outputs a scalar in $[-1, +1]$ via tanh. Two separate networks, matching AlphaGo's architecture.

**한국어**: AlphaGo 형태를 축소한 작은 CNN. 3×3 그리드, 3 입력 채널. 정책 헤드는 9칸에 대한 분포(softmax, 합법수로 마스킹); 가치 헤드는 tanh로 $[-1, +1]$ 스칼라. AlphaGo와 동일하게 두 네트워크 분리.

In [ ]:
class PolicyNet(nn.Module):
    """Small CNN policy for Tic-Tac-Toe."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.head = nn.Linear(32 * 9, 9)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        logits = self.head(h.flatten(start_dim=1))
        return logits  # (B, 9)


class ValueNet(nn.Module):
    """Small CNN value head for Tic-Tac-Toe."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * 9, 64)
        self.head = nn.Linear(64, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        h = F.relu(self.fc(h.flatten(start_dim=1)))
        return torch.tanh(self.head(h)).squeeze(-1)  # (B,)


def policy_probs(net: PolicyNet, state: State) -> np.ndarray:
    """Return a length-9 probability vector over actions, with illegal moves zeroed out."""
    x = state.to_tensor().unsqueeze(0)
    with torch.no_grad():
        logits = net(x).squeeze(0).cpu().numpy()
    mask = np.array([1.0 if c == 0 else 0.0 for c in state.board])
    logits = logits - logits.max()
    exp = np.exp(logits) * mask
    total = exp.sum()
    return exp / total if total > 0 else mask / mask.sum()


def value_estimate(net: ValueNet, state: State) -> float:
    """Return the value from the current player's perspective, in [-1, 1]."""
    x = state.to_tensor().unsqueeze(0)
    with torch.no_grad():
        return net(x).item()


policy_net = PolicyNet()
value_net = ValueNet()
print('PolicyNet params:', sum(p.numel() for p in policy_net.parameters()))
print('ValueNet params :', sum(p.numel() for p in value_net.parameters()))
print('\nUntrained policy probs on empty board:', np.round(policy_probs(policy_net, State.initial()), 3))
print('Untrained value on empty board:', round(value_estimate(value_net, State.initial()), 3))

## Part 4: Supervised Pre-training — Stage 1 of the AlphaGo Pipeline / 지도학습 사전 훈련 — AlphaGo 파이프라인 1단계

**English**: AlphaGo trained $p_\sigma$ on 30M human KGS moves. Here we substitute a surrogate "expert" — an exhaustive **minimax solver** on tic-tac-toe, which computes the provably optimal action for every reachable state. This gives us a clean teacher. The SL policy then learns to imitate these optimal moves via cross-entropy, exactly as in the paper's equation
$$ \Delta\sigma \propto \frac{\partial \log p_\sigma(a\mid s)}{\partial\sigma} $$

**한국어**: AlphaGo는 3천만 KGS 기보로 $p_\sigma$ 를 학습시켰다. 여기서는 대체 "전문가"로 tic-tac-toe의 **전수 minimax solver** 를 쓴다 — 도달 가능한 모든 상태에서 증명 가능한 최적 행동을 계산해준다. SL 정책은 이 최적 수를 cross-entropy로 모방 학습한다 (위 식).

In [ ]:
def minimax_value(state: State, memo=None) -> int:
    """Return the minimax value from the current player's perspective: +1 win, -1 loss, 0 draw."""
    if memo is None:
        memo = {}
    key = (state.board, state.player)
    if key in memo:
        return memo[key]
    w = state.winner()
    if w is not None:
        memo[key] = w * state.player
        return memo[key]
    best = -math.inf
    for m in state.legal_moves():
        v = -minimax_value(state.apply(m), memo)
        if v > best:
            best = v
    memo[key] = best
    return best

def minimax_optimal_moves(state: State, memo) -> list:
    """All actions that achieve the minimax value (there may be ties)."""
    best_val = minimax_value(state, memo)
    return [m for m in state.legal_moves() if -minimax_value(state.apply(m), memo) == best_val]


def enumerate_states():
    """BFS through all non-terminal reachable tic-tac-toe states."""
    seen = set()
    frontier = [State.initial()]
    while frontier:
        s = frontier.pop()
        key = (s.board, s.player)
        if key in seen or s.is_terminal():
            continue
        seen.add(key)
        yield s
        for m in s.legal_moves():
            frontier.append(s.apply(m))

memo = {}
all_states = list(enumerate_states())
print(f'Non-terminal reachable states: {len(all_states)}')

# Build an expert (state, action) dataset: for ties, uniform distribution over optimal moves
X_pol = []
Y_pol = []  # target distribution over 9 cells
for s in all_states:
    opt = minimax_optimal_moves(s, memo)
    target = np.zeros(9, dtype=np.float32)
    for m in opt:
        target[m] = 1.0 / len(opt)
    X_pol.append(s.to_tensor())
    Y_pol.append(torch.from_numpy(target))
X_pol = torch.stack(X_pol)
Y_pol = torch.stack(Y_pol)
print('SL dataset:', X_pol.shape, Y_pol.shape)

In [ ]:
# Stage 1: train the SL policy with cross-entropy against the optimal-move distribution
policy_net = PolicyNet()
opt = torch.optim.Adam(policy_net.parameters(), lr=5e-3)
losses = []
for epoch in range(300):
    perm = torch.randperm(X_pol.shape[0])
    total = 0.0
    for i in range(0, X_pol.shape[0], 64):
        idx = perm[i:i+64]
        logits = policy_net(X_pol[idx])
        # KL = -sum(target * log_softmax(logits))
        logp = F.log_softmax(logits, dim=-1)
        loss = -(Y_pol[idx] * logp).sum(dim=-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * idx.shape[0]
    losses.append(total / X_pol.shape[0])

plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy loss')
plt.title('SL policy training — imitating minimax-optimal moves / 최적 수 모방 학습')
plt.grid(alpha=0.3); plt.show()

# Measure top-1 agreement with the expert (AlphaGo's headline number was 57%)
policy_net.eval()
agree = 0
for s in all_states:
    probs = policy_probs(policy_net, s)
    predicted = int(np.argmax(probs))
    if predicted in minimax_optimal_moves(s, memo):
        agree += 1
print(f'\nSL policy top-1 agreement with optimal play: {agree}/{len(all_states)} = {agree/len(all_states):.1%}')
print('(AlphaGo paper: 57.0% top-1 on KGS human moves)')

## Part 5: PUCT — Adding the Policy Prior to MCTS / MCTS에 정책 prior 추가

**English**: Replace the UCT exploration term with AlphaGo's PUCT formulation (paper equation):
$$ a_t = \arg\max_a \big[Q(s,a) + c_{\text{puct}}\, P(s,a)\, \tfrac{\sqrt{N(s)}}{1 + N(s,a)}\big] $$
where $P(s,a) = p_\sigma(a\mid s)$. Search now concentrates on moves the policy considers plausible — dramatically fewer useless branches are explored.

**한국어**: UCT의 exploration 항을 AlphaGo의 PUCT로 바꾼다 (논문 식). prior $P(s,a) = p_\sigma(a\mid s)$ 로 정책이 유망하다고 보는 수에 탐색이 집중된다 — 쓸모없는 가지 탐색이 급감한다.

In [ ]:
def puct_select(node: Node, c_puct: float = 1.5) -> Node:
    """AlphaGo-style PUCT child selection."""
    sqrt_sum_N = math.sqrt(node.N + 1)
    best, best_val = None, -math.inf
    for action, child in node.children.items():
        P = node.prior[action]
        u = c_puct * P * sqrt_sum_N / (1 + child.N)
        Q = -child.Q  # opponent perspective
        val = Q + u
        if val > best_val:
            best, best_val = child, val
    return best


def expand_with_prior(node: Node, policy: PolicyNet):
    """Expand all children, with priors from the policy network."""
    probs = policy_probs(policy, node.state)
    for m in node.state.legal_moves():
        node.children[m] = Node(state=node.state.apply(m), parent=node, move_from_parent=m)
        node.prior[m] = float(probs[m])


def mcts_alphago(root_state: State, policy: PolicyNet, value: ValueNet,
                 n_simulations: int = 500, lam: float = 0.5) -> tuple:
    """AlphaGo-style MCTS: PUCT selection, policy prior, value-net + rollout mix at leaves."""
    root = Node(state=root_state)
    expand_with_prior(root, policy)
    for _ in range(n_simulations):
        node = root
        while node.is_expanded() and not node.state.is_terminal():
            node = puct_select(node)
        if not node.state.is_terminal():
            expand_with_prior(node, policy)
            # Leaf evaluation: (1-lam) * v_theta + lam * z (rollout outcome)
            v_net = value_estimate(value, node.state)
            if lam > 0:
                z = random_rollout(node.state) * node.state.player  # perspective fix
            else:
                z = 0.0
            V = (1 - lam) * v_net + lam * z
        else:
            V = node.state.winner() * node.state.player
        backup(node, V)
    # Return most-visited action AND the visit distribution (for self-play training)
    visits = {a: c.N for a, c in root.children.items()}
    best_move = max(visits, key=visits.get)
    return best_move, visits


# Contrast: same state, plain UCT vs AlphaGo-style MCTS
s = State.initial().apply(4)  # X plays centre, O to move
move_uct = mcts_uct(s, n_simulations=1000)
move_ag, visits = mcts_alphago(s, policy_net, value_net, n_simulations=1000, lam=0.5)
print(f'UCT move (1000 sims):         cell {move_uct}')
print(f'AlphaGo-style move (lam=0.5): cell {move_ag}')
print(f'Visit distribution: {sorted(visits.items(), key=lambda kv: -kv[1])}')
print('Optimal responses to centre: any corner')

## Part 6: Value Network Training with One-Position-Per-Game / "대국당 한 국면" 전략으로 가치망 학습

**English**: The paper's most important methodological caveat: training $v_\theta$ on consecutive positions from whole self-play games **causes severe overfitting** (train MSE 0.19, test MSE 0.37). The fix is to sample **one position per game**. We reproduce this in miniature by generating self-play games with the current policy and comparing the two sampling regimes.

**한국어**: 논문의 가장 중요한 방법론적 주의사항: 완전 대국의 연속 국면으로 $v_\theta$ 를 학습하면 **심각한 과적합** 이 발생한다 (훈련 MSE 0.19, 테스트 MSE 0.37). 해결책은 **대국당 한 국면** 만 샘플링. self-play로 두 방식을 축소판에서 재현한다.

In [ ]:
def self_play_game(policy: PolicyNet, stochastic: bool = True, eps: float = 0.15):
    """Play one self-play game with the policy. Return list of states and the terminal outcome from +1's perspective."""
    s = State.initial()
    trajectory = []
    while not s.is_terminal():
        probs = policy_probs(policy, s)
        if stochastic and random.random() < eps:
            m = random.choice(s.legal_moves())
        else:
            m = int(np.random.choice(9, p=probs))
        trajectory.append(s)
        s = s.apply(m)
    return trajectory, s.winner()


def build_value_dataset(policy: PolicyNet, n_games: int, mode: str):
    """
    mode='all'    → all positions from every game (naive, correlated).
    mode='single' → one random position per game (AlphaGo's trick).
    """
    X, Y = [], []
    for _ in range(n_games):
        traj, z = self_play_game(policy)
        if mode == 'all':
            for s in traj:
                X.append(s.to_tensor())
                Y.append(float(z * s.player))  # value from player-to-move perspective
        else:
            s = random.choice(traj)
            X.append(s.to_tensor())
            Y.append(float(z * s.player))
    return torch.stack(X), torch.tensor(Y, dtype=torch.float32)


def train_value_net(X_train, Y_train, X_test, Y_test, epochs: int = 200, lr: float = 3e-3):
    net = ValueNet()
    optz = torch.optim.Adam(net.parameters(), lr=lr)
    tr_hist, te_hist = [], []
    for _ in range(epochs):
        perm = torch.randperm(X_train.shape[0])
        for i in range(0, X_train.shape[0], 64):
            idx = perm[i:i+64]
            pred = net(X_train[idx])
            loss = F.mse_loss(pred, Y_train[idx])
            optz.zero_grad(); loss.backward(); optz.step()
        with torch.no_grad():
            tr_hist.append(F.mse_loss(net(X_train), Y_train).item())
            te_hist.append(F.mse_loss(net(X_test), Y_test).item())
    return net, tr_hist, te_hist


N_GAMES = 800

print('Generating value-training datasets via self-play...')
X_all, Y_all = build_value_dataset(policy_net, n_games=N_GAMES, mode='all')
X_sin, Y_sin = build_value_dataset(policy_net, n_games=N_GAMES, mode='single')
# Use a held-out test set of single-position-per-game for both (i.i.d. held-out positions)
X_te, Y_te  = build_value_dataset(policy_net, n_games=300, mode='single')

print(f'All-positions dataset size:        {X_all.shape[0]}')
print(f'One-per-game dataset size:         {X_sin.shape[0]}')
print(f'Held-out test size:                {X_te.shape[0]}')

_, tr_all, te_all = train_value_net(X_all, Y_all, X_te, Y_te)
_, tr_sin, te_sin = train_value_net(X_sin, Y_sin, X_te, Y_te)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].plot(tr_all, label='Train MSE'); axes[0].plot(te_all, label='Test MSE', ls='--')
axes[0].set_title(f'"All positions" (correlated) / 전체 국면 (상관됨)\nFinal train={tr_all[-1]:.3f}, test={te_all[-1]:.3f}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE'); axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(tr_sin, label='Train MSE'); axes[1].plot(te_sin, label='Test MSE', ls='--')
axes[1].set_title(f'"One per game" (AlphaGo fix) / 대국당 하나 (AlphaGo 해법)\nFinal train={tr_sin[-1]:.3f}, test={te_sin[-1]:.3f}')
axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3); axes[1].legend()
plt.suptitle('Overfitting diagnosis — reproducing AlphaGo Fig/text: 0.19 vs 0.37 → 0.226 vs 0.234')
plt.tight_layout(); plt.show()

print(f'\nGap (test - train) for naive method: {te_all[-1] - tr_all[-1]:+.3f}')
print(f'Gap for "one per game" method:      {te_sin[-1] - tr_sin[-1]:+.3f}')
print('The second should be much smaller — same lesson as the paper.')

## Part 7: The $\lambda$ Ablation — Reproducing Fig. 4b / $\lambda$ Ablation — Fig 4b 재현

**English**: The headline ablation in the paper: sweep $\lambda \in \{0, 0.25, 0.5, 0.75, 1.0\}$ and measure playing strength. $\lambda=0$ = value-network only; $\lambda=1$ = rollout only; the paper reports $\lambda=0.5$ winning ≥95% against all other variants. We reproduce the pattern at miniature scale by playing many games between each variant and a fixed plain-UCT opponent, and by pairing variants head-to-head.

**한국어**: 논문의 간판 ablation: $\lambda \in \{0, 0.25, 0.5, 0.75, 1.0\}$ 를 스윕하여 기력 측정. $\lambda=0$ = 가치망만; $\lambda=1$ = 롤아웃만; 논문은 $\lambda=0.5$ 가 다른 모든 변형 상대 ≥95% 승률. 축소판에서 각 변형 vs 고정 UCT 및 변형끼리의 head-to-head 로 패턴을 재현.

In [ ]:
# Train a good value net on the 'one per game' dataset (the AlphaGo fix)
good_value_net, _, _ = train_value_net(X_sin, Y_sin, X_te, Y_te, epochs=200)

def play_one(agent_x, agent_o) -> int:
    """Play one game. agent_x plays first as +1, agent_o as -1. Return +1/-1/0."""
    s = State.initial()
    while not s.is_terminal():
        mover = agent_x if s.player == +1 else agent_o
        m = mover(s)
        s = s.apply(m)
    return s.winner()


def make_alphago_agent(policy, value, lam, n_sim=200):
    return lambda s: mcts_alphago(s, policy, value, n_simulations=n_sim, lam=lam)[0]

def make_uct_agent(n_sim=200):
    return lambda s: mcts_uct(s, n_simulations=n_sim)


def score_match(agent_a, agent_b, n_games: int = 40) -> float:
    """Return win rate of agent_a (with ties counted as 0.5), alternating colours."""
    wins = 0.0
    for i in range(n_games):
        if i % 2 == 0:
            r = play_one(agent_a, agent_b)
            wins += 1.0 if r == +1 else 0.5 if r == 0 else 0.0
        else:
            r = play_one(agent_b, agent_a)
            wins += 1.0 if r == -1 else 0.5 if r == 0 else 0.0
    return wins / n_games


lambdas = [0.0, 0.25, 0.5, 0.75, 1.0]
uct_opponent = make_uct_agent(n_sim=200)
win_vs_uct = {}
for lam in lambdas:
    agent = make_alphago_agent(policy_net, good_value_net, lam, n_sim=200)
    win_vs_uct[lam] = score_match(agent, uct_opponent, n_games=40)
    print(f'lam={lam:.2f} win rate vs UCT-200: {win_vs_uct[lam]:.2f}')

In [ ]:
# Head-to-head tournament between lam variants (mirrors Fig. 4b)
agents = {lam: make_alphago_agent(policy_net, good_value_net, lam, n_sim=200) for lam in lambdas}
matrix = np.zeros((len(lambdas), len(lambdas)))
for i, la in enumerate(lambdas):
    for j, lb in enumerate(lambdas):
        if i == j:
            matrix[i, j] = 0.5
        else:
            matrix[i, j] = score_match(agents[la], agents[lb], n_games=20)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(lambdas))); ax.set_yticks(range(len(lambdas)))
ax.set_xticklabels([f'λ={l}' for l in lambdas]); ax.set_yticklabels([f'λ={l}' for l in lambdas])
ax.set_xlabel('Opponent'); ax.set_ylabel('Player')
for i in range(len(lambdas)):
    for j in range(len(lambdas)):
        ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center', color='black')
plt.colorbar(im, ax=ax, label='Win rate of row player')
plt.title('Head-to-head win rates by λ (miniature Fig. 4b) / λ별 head-to-head 승률')
plt.tight_layout(); plt.show()

overall = matrix.mean(axis=1)
print('\nOverall strength (mean row score):')
for l, s in zip(lambdas, overall):
    print(f'  λ={l}: {s:.3f}')
print('\nPaper: λ=0.5 dominates ≥95% against variants. Our miniature should also peak near λ=0.5.')

## Part 8: Policy-Gradient Self-Play (REINFORCE) — Stage 2 of the Pipeline / Policy gradient self-play — 파이프라인 2단계

**English**: Reproducing the RL stage in miniature: take the SL policy $p_\sigma$ and improve it with REINFORCE against a frozen opponent pool of prior snapshots (the paper's anti-overfitting trick). Because tic-tac-toe is already solved by our SL policy in Part 4, we start from a **weakened random policy** here so there is room to improve. The update rule is exactly the paper's:
$$ \Delta\rho \propto \nabla_\rho \log p_\rho(a_t\mid s_t)\, z_t $$

**한국어**: RL 단계의 축소판 재현: SL 정책 $p_\sigma$ 를 REINFORCE로 개선하되, 과거 스냅샷 풀(논문의 과적합 방지 비결)과 대결. 우리 SL 정책은 이미 Part 4에서 최적이라 개선 여지가 없으므로 **약화된 랜덤 정책** 에서 출발한다. 업데이트 규칙은 논문과 동일.

In [ ]:
def play_against_pool(rl_net, opponent_nets):
    """One game; rl_net is +1, a random snapshot from opponent_nets plays -1."""
    opp = random.choice(opponent_nets)
    s = State.initial()
    log_probs = []
    while not s.is_terminal():
        if s.player == +1:
            logits = rl_net(s.to_tensor().unsqueeze(0)).squeeze(0)
            mask = torch.tensor([0.0 if c == 0 else -1e9 for c in s.board])
            probs = F.softmax(logits + mask, dim=-1)
            a = int(torch.multinomial(probs, 1).item())
            log_probs.append(torch.log(probs[a] + 1e-9))
        else:
            probs = policy_probs(opp, s)
            a = int(np.random.choice(9, p=probs))
        s = s.apply(a)
    return s.winner(), log_probs


# Weak starting point — randomly initialised policy
rl_policy = PolicyNet()
import copy
opponent_pool = [copy.deepcopy(rl_policy)]
opt = torch.optim.Adam(rl_policy.parameters(), lr=5e-3)

win_history = []
for iteration in range(150):
    rl_policy.train()
    batch_loss = 0.0
    batch_wins = 0
    for _ in range(16):
        z, log_probs = play_against_pool(rl_policy, opponent_pool)
        if log_probs:
            loss = -torch.stack(log_probs).sum() * z  # REINFORCE: ∇log p · z
            batch_loss = batch_loss + loss
        batch_wins += 1 if z == +1 else (0.5 if z == 0 else 0)
    batch_loss = batch_loss / 16
    opt.zero_grad(); batch_loss.backward(); opt.step()
    win_history.append(batch_wins / 16)
    # Freeze a snapshot every 15 iterations (like the paper's pool of prior iterations)
    if iteration % 15 == 14:
        opponent_pool.append(copy.deepcopy(rl_policy))

plt.plot(win_history, alpha=0.7)
plt.xlabel('Iteration'); plt.ylabel('Win rate vs opponent pool')
plt.title('RL policy self-play training / RL 정책 self-play 학습')
plt.axhline(0.5, color='gray', ls='--', alpha=0.5)
plt.grid(alpha=0.3); plt.show()

print(f'Opponent pool size at end: {len(opponent_pool)}')
print(f'Final win rate vs pool: {np.mean(win_history[-10:]):.2%}')
print('AlphaGo: p_rho wins >80% vs p_sigma, 85% vs Pachi (no search).')

## Summary / 요약

| Concept / 개념 | AlphaGo paper (2016) / 본 논문 | This notebook / 이 노트북 |
|---|---|---|
| **Domain** | 19×19 Go, $b^d\approx 250^{150}$ | Tic-tac-toe 3×3, solvable exactly |
| **SL policy $p_\sigma$** | 13-layer CNN, 192 filters, 30M KGS moves, 57.0% top-1 | 2-conv CNN, minimax-optimal teacher, ~100% on reachable states |
| **Rollout policy $p_\pi$** | Linear softmax on patterns, 24.2%, 2 μs | Uniform random (same role: fast) |
| **RL policy $p_\rho$** | REINFORCE self-play vs pool; beats $p_\sigma$ 80%, Pachi 85% | REINFORCE with frozen snapshot pool; win-rate rises over iterations |
| **Value network $v_\theta$** | CNN regressor, 30M **one-per-game** positions, MSE 0.226/0.234 | CNN regressor, 'one-per-game' vs 'all' comparison; overfitting reproduced |
| **MCTS selection** | PUCT $Q + c_{\text{puct}} P \sqrt{N}/(1+N(s,a))$ | Same formula, $c_{\text{puct}}=1.5$ |
| **Leaf evaluation** | $V(s_L)=(1-\lambda) v_\theta + \lambda z_L$, $\lambda=0.5$ | Same; $\lambda$-sweep shows interior mixtures dominate |
| **Compute** | 40 search threads, 1,202 CPU, 176 GPU (distributed) | Single CPU, 200 simulations per move |
| **Playing strength** | 3,140 Elo; beat Fan Hui 5–0 | Beats plain UCT of matched sims with $\lambda\approx 0.5$ |

### Key lessons reproduced / 재현된 핵심 교훈

1. **PUCT concentrates search on plausible moves** — demonstrated in Part 5. / PUCT는 유망한 수에 탐색을 집중시킨다 (Part 5).
2. **"One position per game" resolves value-net overfitting** — Part 6 reproduces the 0.19/0.37 → 0.226/0.234 gap collapse. / '대국당 한 국면' 으로 가치망 과적합 해결 (Part 6).
3. **Mixed leaf evaluation ($\lambda \approx 0.5$) dominates pure variants** — Part 7 reproduces the Fig. 4b pattern. / 혼합 리프 평가가 순수 변형을 지배 (Part 7, Fig 4b 재현).
4. **Policy gradient with a frozen opponent pool stabilises self-play** — Part 8 shows monotonic win-rate growth. / 과거 스냅샷 풀과의 policy gradient가 self-play 를 안정화 (Part 8).

The full AlphaGo's empirical strength is out of reach, but every **algorithmic idea** in the paper is present, testable, and behaves qualitatively as the paper reports. / AlphaGo의 실제 기력 수준은 이 노트북으로 재현 불가능하지만, 논문의 모든 **알고리즘 아이디어** 가 포함되어 있고, 테스트 가능하며, 논문이 보고한 대로 정성적으로 동작한다.